# Séance 3 — Analyse exploratoire & visualisation

**Decision problem:** what pattern is strong enough to change what we do next?

Official topic preserved: EDA and visualization. The output is a reusable analysis summary and chart.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]:
    d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"

def load_raw_trends():
    p = DATA / "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"])
    for c in ["chatgpt", "iphone", "meteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)

def load_clean_long():
    p = OUT / "bloc1" / "clean_trends_long.csv"
    if p.exists():
        df = pd.read_csv(p, parse_dates=["date"])
    else:
        raw = load_raw_trends()
        df = raw.melt(id_vars="date", var_name="signal", value_name="interest")
    return df.sort_values(["date", "signal"]).reset_index(drop=True)

In [ ]:
df = load_clean_long()
df["rolling_4w"] = df.groupby("signal")["interest"].transform(lambda s: s.rolling(4, min_periods=1).mean())
summary = df.groupby("signal").agg(rows=("interest","size"), average=("interest","mean"), median=("interest","median"), peak=("interest","max"), latest=("interest","last")).round(2)
summary["latest_vs_median"] = (summary["latest"] - summary["median"]).round(2)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
for signal, g in df.groupby("signal"):
    ax.plot(g["date"], g["rolling_4w"], label=signal)
ax.set_title("Weak signals: 4-week rolling attention")
ax.set_ylabel("Google Trends index")
ax.legend()
fig.tight_layout()
chart = OUT / "bloc1" / "eda_weak_signals.png"
fig.savefig(chart, dpi=140)
summary.to_csv(OUT / "bloc1" / "eda_signal_summary.csv")
note = "Recommendation: monitor ChatGPT as a strategic weak signal, use weather as a seasonal control, and iPhone as an event-driven benchmark."
(OUT / "bloc1" / "eda_decision_note.md").write_text(note + "\n")
print("Saved", chart)

## Practical exercise

Write one decision that the chart supports and one decision it does not support.

## Conclusion

EDA converts clean data into evidence, but only if it ends with interpretation and limits.